# RAG Evaluation mit DeepEval

Dieses Notebook evaluiert die RAG-Pipeline mit **DeepEval** und **GPT-4o-mini** als Judge-Modell.

### Metriken
| Metrik | Was wird gemessen | Referenzantwort nötig? |
|---|---|---|
| **Faithfulness** | Erfindet das LLM Infos die nicht im Kontext stehen? | Nein |
| **Answer Relevancy** | Beantwortet die Antwort wirklich die Frage? | Nein |
| **Contextual Precision** | Sind die retrieved Chunks tatsächlich relevant? | Nein |
| **Contextual Recall** | Wurden alle relevanten Stellen gefunden? | Ja |

### Kategorien im Fragenkatalog
- **A** – Faktische Detail- und Datenextraktion
- **B** – Methoden- und Begründungsverständnis  
- **C** – Cross-Document-Synthese
- **D** – Struktur- und Orientierungsfragen
- **E** – Anti-Halluzination und Unsicherheitsverhalten
- **F** – Domänenwissen-Transfer

In [1]:
pip install deepeval openai python-dotenv


  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/844.6 kB ? eta -:--:--
   ---------------------------------------- 844.6/844.6 kB 7.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 9.6 MB/s  0:00:00
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -------------- ------------------------- 3.4/9.5 MB 16.7 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.5 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 17.3 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 14.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 11.0 MB/s  0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ----------------------------------------  0/16 [pywin32]
   -------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.5.1 requires posthog<6.0.0,>=2.4.0, but you have posthog 7.12.0 which is incompatible.


In [ ]:
# Setup: Projekt-Root setzen
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print('ROOT:', ROOT)

In [ ]:
# API-Key laden
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import openai
if not os.getenv('OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY') == 'your_api_key_here':
    raise ValueError('OPENAI_API_KEY nicht gesetzt! Bitte in .env eintragen.')

print('API-Key geladen.')

In [ ]:
# Konfiguration

# RAG-Einstellungen
RAG_MODE       = 'hybrid'
RAG_K          = 20
CHUNK_SIZE     = 1200
CHUNK_OVERLAP  = 200
USE_RERANKER   = True
RERANK_TOP_N   = 5

# Judge-Modell (GPT-4o-mini ist günstiger, für Bachelorarbeit ausreichend)
JUDGE_MODEL = 'gpt-4o-mini'   # Alternativ: 'gpt-4o'

# Welche Fragen evaluieren? None = alle 37
# Beispiel für schnellen Test: QUESTION_IDS = ['q1', 'q2', 'q3']
QUESTION_IDS = None

# Schwellwert ab dem eine Metrik als bestanden gilt (0.0 – 1.0)
THRESHOLD = 0.7

print(f'Judge: {JUDGE_MODEL} | RAG: {RAG_MODE}, k={RAG_K}, reranker={USE_RERANKER}, top_n={RERANK_TOP_N}')
print(f'Threshold: {THRESHOLD} | Fragen: {"alle" if QUESTION_IDS is None else QUESTION_IDS}')

In [ ]:
# Fragen laden
from src.evaluation import load_questions

QUESTIONS_PATH = ROOT / 'data' / 'eval' / 'questions.jsonl'
all_questions = load_questions(QUESTIONS_PATH)

if QUESTION_IDS is not None:
    questions = [q for q in all_questions if q['id'] in QUESTION_IDS]
else:
    questions = all_questions

print(f'{len(questions)} Fragen geladen.')
for q in questions:
    print(f"  [{q['id']}] Kategorie {q['category']}: {q['query'][:70]}...")

In [ ]:
# RAG für alle Fragen ausführen und Ergebnisse sammeln
# Hinweis: Dieser Schritt läuft lokal (Ollama + Reranker) und dauert je nach
# Hardware ca. 30–120 Sekunden pro Frage.
import time
from src.evaluation import run_rag_for_eval

rag_results = []

for i, q in enumerate(questions, start=1):
    print(f'[{i}/{len(questions)}] {q["id"]} ({q["category"]}): {q["query"][:60]}...')
    t0 = time.perf_counter()

    result = run_rag_for_eval(
        query=q['query'],
        mode=RAG_MODE,
        k=RAG_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        use_reranker=USE_RERANKER,
        rerank_top_n=RERANK_TOP_N,
    )

    dt = time.perf_counter() - t0
    rag_results.append({
        'id': q['id'],
        'category': q['category'],
        'query': q['query'],
        'expected_output': q.get('expected_output', ''),
        'actual_output': result['answer'],
        'retrieval_context': result['retrieval_context'],
        'sources': result['sources'],
        'latency_s': round(dt, 2),
    })
    print(f'  -> {dt:.1f}s | Antwort: {result["answer"][:100]}...')

print(f'\nAlle {len(rag_results)} RAG-Läufe abgeschlossen.')

In [ ]:
# DeepEval Test Cases erstellen
from deepeval.test_case import LLMTestCase

test_cases = []
for r in rag_results:
    tc = LLMTestCase(
        input=r['query'],
        actual_output=r['actual_output'],
        expected_output=r['expected_output'],
        retrieval_context=r['retrieval_context'],
    )
    test_cases.append(tc)

print(f'{len(test_cases)} Test Cases erstellt.')

In [ ]:
# Metriken konfigurieren mit GPT-4o-mini als Judge
from deepeval.models import GPTModel
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

judge = GPTModel(model=JUDGE_MODEL)

metrics = [
    FaithfulnessMetric(model=judge, threshold=THRESHOLD),
    AnswerRelevancyMetric(model=judge, threshold=THRESHOLD),
    ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
    ContextualRecallMetric(model=judge, threshold=THRESHOLD),
]

print('Metriken konfiguriert:', [type(m).__name__ for m in metrics])

In [ ]:
# Evaluation ausführen
# Hinweis: Dieser Schritt sendet Anfragen an die OpenAI API (GPT-4o-mini als Judge).
# Kosten: ca. $0.50–2.00 für alle 37 Fragen mit 4 Metriken.
from deepeval import evaluate

eval_results = evaluate(test_cases, metrics=metrics, print_results=False)
print('Evaluation abgeschlossen.')

In [ ]:
# Ergebnisse pro Frage anzeigen
import pandas as pd

rows = []
for r, tc in zip(rag_results, eval_results.test_results):
    row = {
        'ID': r['id'],
        'Kategorie': r['category'],
        'Frage (kurz)': r['query'][:55] + '...',
        'Latenz (s)': r['latency_s'],
    }
    for metric_result in tc.metrics_data:
        name = type(metric_result).__name__.replace('Metric', '').replace('Contextual', 'Ctx')
        row[name] = round(metric_result.score, 2) if metric_result.score is not None else None
        row[name + ' ✓'] = '✓' if metric_result.success else '✗'
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Zusammenfassung nach Kategorie
print('=== Durchschnitt nach Kategorie ===')
metric_cols = [c for c in df.columns if not c.endswith('✓') and c not in ('ID', 'Kategorie', 'Frage (kurz)', 'Latenz (s)')]
summary = df.groupby('Kategorie')[metric_cols].mean().round(2)
print(summary.to_string())

print('\n=== Gesamt-Durchschnitt ===')
print(df[metric_cols].mean().round(2).to_string())

print(f'\n=== Bestandsrate (Threshold={THRESHOLD}) ===')
pass_cols = [c for c in df.columns if c.endswith('✓')]
for col in pass_cols:
    passed = (df[col] == '✓').sum()
    print(f'  {col}: {passed}/{len(df)} ({100*passed/len(df):.0f}%)')